# **Apriori Analysis on Pakistan Traffic Accidents Dataset**

This repository implements the **Apriori Analysis** on a dataset of traffic accidents in Pakistan. The dataset contains only **64 rows** of data in a CSV file, which limits the depth of analysis. Due to the small dataset size, the process generates candidate itemsets only up to **Candidate 2 (C2)**.

The analysis identifies frequent patterns and associations between attributes like **Fatal Accidents**, **Non-Fatal Accidents**, **Number of Vehicles Involved**, and more. By applying thresholds like minimum support, the frequent itemsets provide insights into recurring patterns in traffic accidents.

This project serves as a practical example of using the Apriori algorithm on smaller datasets, demonstrating how to handle and analyze limited data effectively.

# **Formulas Used in Apriori Analysis**

## **1. Candidate Itemset Generation**
The number of candidate \( n \)-itemsets generated at each step is determined by the combination of frequent \((n-1)\)-itemsets:


**|Cn| = |L(n-1)|! / (n! * (|L(n-1)| - n)!)**

Where:
- |Cn|: Number of candidate n-itemsets.
- |L(n-1)|: Number of frequent (n-1)-itemsets.

---

## **2. Pruning Step**
Based on the **Apriori Property**, an itemset is pruned if any of its subsets is not frequent:

**If subset(X) is not frequent, then X is not frequent.**

- X: Candidate itemset.

---

## **3. Support**
Support measures the frequency of an itemset appearing in transactions:

**Support(X) = (Number of Transactions Containing X) / (Total Number of Transactions)**

---

## **4. Confidence**
Confidence measures the strength of an association rule:

**Confidence(X => Y) = Support(X ∪ Y) / Support(X)**

- X: Antecedent (if-part of the rule).
- Y: Consequent (then-part of the rule).
---

## **5. Lift**
Lift measures the strength of a rule compared to random chance:

**Lift(X => Y) = Confidence(X => Y) / Support(Y)**


Where:
- Lift > 1: Positive correlation between X and Y.
- Lift = 1: No correlation between X and Y.
- Lift < 1: Negative correlation between X and Y.

---

## **Summary Table of Metrics**

| **Metric**       | **Formula**                                       | **Purpose**                               |
|-------------------|---------------------------------------------------|-------------------------------------------|
| **Support**       | ( Support(X) = {{Transactions Containing } X}/{{Total Transactions}} ) | Measures frequency of an itemset          |
| **Confidence**    | ( Confidence(X -> Y) = Support(X U Y)/Support(X) ) | Measures strength of an association rule |
| **Lift**          | ( Lift(X -> Y) = Confidence}(X -> Y)/Support(Y) ) | Measures correlation strength            |
| **Candidate Size**| Cn = (L(n-1))! / (n! * (L(n-1) - n)!)                 | Number of generated itemsets at step \( n \) |

In [ ]:
import pandas as pd
from itertools import combinations
from IPython.display import display, HTML

# Load Dataset
data = pd.read_csv("Datasets/pak-traffic-accidents-annual.csv")

# Define thresholds for categorization
thresholds = {
    "Total number of accidents": 5000,
    "Fatal Accidents": 2000,
    "Non-Fatal Accidents": 3000,
    "Killed": 3000,
    "Injured": 8000,
    "Total number of vehicles involved": 5000
}

# Categorize data into binary transactions
def categorize(row, thresholds):
    transaction = []
    for col, threshold in thresholds.items():
        if row[col] > threshold:
            transaction.append(f"{col} > {threshold}")
        else:
            transaction.append(f"{col} <= {threshold}")
    return transaction

# Generate transactions
transactions = data.apply(lambda row: categorize(row, thresholds), axis=1).tolist()

# Generate C1
def generate_c1(transactions):
    c1 = {}
    for transaction in transactions:
        for item in transaction:
            c1[item] = c1.get(item, 0) + 1
    return c1

# Generate Cn from Ln-1
def generate_cn(ln_minus_1):
    ln_items = list(ln_minus_1.keys())  # Get keys from L(n-1)
    candidates = []
    for i in range(len(ln_items)):
        for j in range(i + 1, len(ln_items)):
            # Combine two itemsets if possible
            candidate = tuple(sorted(set([ln_items[i], ln_items[j]])))  # Ensure it's a valid tuple
            candidates.append(candidate)
    return candidates


# Count support for Cn
def count_support(candidates, transactions):
    support_counts = {itemset: 0 for itemset in candidates}
    for transaction in transactions:
        for itemset in candidates:
            if set(itemset).issubset(transaction):
                support_counts[itemset] += 1
    
    print(f"Support Counts for Candidates: {support_counts}")
    return support_counts

# Filter Cn to generate Ln
def generate_ln(cn_support, min_support):
    return {itemset: support for itemset, support in cn_support.items() if support >= min_support}

# Apriori Algorithm with Dynamic Minimum Support
frequent_itemsets = {}
candidate_sets = {}
support_data = {}
current_level = 1

# Generate C1
c1 = generate_c1(transactions)
candidate_sets[current_level] = c1

# Iterative generation of Ln and Cn
while True:
    c_df = pd.DataFrame({
        "Itemset": [' & '.join(map(str, itemset)) if isinstance(itemset, tuple) else itemset for itemset in candidate_sets[current_level].keys()],
        "Support Count": list(candidate_sets[current_level].values())
    })
    display(HTML(c_df.to_html(index=False)))

    # Prompt user for minimum support
    min_support = int(input(f"Enter minimum support for L{current_level}: "))

    # Generate Ln
    ln = generate_ln(candidate_sets[current_level], min_support)
    l_df = pd.DataFrame({
        "Itemset": [' & '.join(itemset) if isinstance(itemset, tuple) else itemset for itemset in ln.keys()],
        "Support Count": list(ln.values())
    })
    display(HTML(l_df.to_html(index=False)))

    if not ln:
        print(f"No frequent itemsets found at level {current_level}. Stopping analysis.")
        break

    # Store Ln and prepare for next level
    frequent_itemsets[current_level] = ln
    support_data.update(ln)

    # Generate Cn+1
    cn = generate_cn(ln)
    if not cn:
        print(f"No candidates generated for level {current_level + 1}. Stopping analysis.")
        break

    # Count support for Cn+1
    cn_support = count_support(cn, transactions)
    current_level += 1
    candidate_sets[current_level] = cn_support

# Final Results
print("Final Frequent Itemsets:")
for level, itemsets in frequent_itemsets.items():
    print(f"L{level}:")
    df = pd.DataFrame({
        "Itemset": [' & '.join(itemset) if isinstance(itemset, tuple) else itemset for itemset in itemsets.keys()],
        "Support Count": list(itemsets.values())
    })
    display(HTML(df.to_html(index=False)))

Candidate C1 (Generated):


Itemset,Support Count
Total number of accidents > 5000,15
Fatal Accidents > 2000,18
Non-Fatal Accidents > 3000,15
Killed > 3000,15
Injured > 8000,11
Total number of vehicles involved > 5000,19
Non-Fatal Accidents <= 3000,47
Killed <= 3000,47
Injured <= 8000,51
Total number of accidents <= 5000,47


L1 (Frequent 1-itemsets):


Itemset,Support Count
Non-Fatal Accidents <= 3000,47
Killed <= 3000,47
Injured <= 8000,51
Total number of accidents <= 5000,47
Total number of vehicles involved <= 5000,43
Fatal Accidents <= 2000,44


Support Counts for Candidates: {('Killed <= 3000', 'Non-Fatal Accidents <= 3000'): 43, ('Injured <= 8000', 'Non-Fatal Accidents <= 3000'): 47, ('Non-Fatal Accidents <= 3000', 'Total number of accidents <= 5000'): 43, ('Non-Fatal Accidents <= 3000', 'Total number of vehicles involved <= 5000'): 43, ('Fatal Accidents <= 2000', 'Non-Fatal Accidents <= 3000'): 40, ('Injured <= 8000', 'Killed <= 3000'): 47, ('Killed <= 3000', 'Total number of accidents <= 5000'): 46, ('Killed <= 3000', 'Total number of vehicles involved <= 5000'): 42, ('Fatal Accidents <= 2000', 'Killed <= 3000'): 44, ('Injured <= 8000', 'Total number of accidents <= 5000'): 47, ('Injured <= 8000', 'Total number of vehicles involved <= 5000'): 43, ('Fatal Accidents <= 2000', 'Injured <= 8000'): 44, ('Total number of accidents <= 5000', 'Total number of vehicles involved <= 5000'): 43, ('Fatal Accidents <= 2000', 'Total number of accidents <= 5000'): 44, ('Fatal Accidents <= 2000', 'Total number of vehicles involved <= 5000'

...

Itemset,Support Count


No frequent itemsets found at level 3. Stopping analysis.
Final Frequent Itemsets:
L1:


Itemset,Support Count
Non-Fatal Accidents <= 3000,47
Killed <= 3000,47
Injured <= 8000,51
Total number of accidents <= 5000,47
Total number of vehicles involved <= 5000,43
Fatal Accidents <= 2000,44


L2:


Itemset,Support Count
Injured <= 8000 & Non-Fatal Accidents <= 3000,47
Injured <= 8000 & Killed <= 3000,47
Killed <= 3000 & Total number of accidents <= 5000,46
Injured <= 8000 & Total number of accidents <= 5000,47


In [62]:
from itertools import combinations

support_data = {}
for level, itemsets in frequent_itemsets.items():
    # Convert itemsets to tuples of meaningful strings
    itemsets_as_tuples = {
        tuple([item]) if isinstance(item, str) else tuple(sorted(item)): support
        for item, support in itemsets.items()
    }
    support_data.update(itemsets_as_tuples)

print(frequent_itemsets)

# Function to calculate confidence for rules
def calculate_confidence(frequent_itemsets, support_data):
    """
    Calculate confidence for association rules from frequent itemsets.

    Parameters:
        frequent_itemsets (dict): Dictionary of frequent itemsets (Ln).
        support_data (dict): Support counts for all itemsets.

    Returns:
        list: A list of rules with confidence values.
    """
    rules = []
    for level, itemsets in frequent_itemsets.items():
        for itemset, support in itemsets.items():
            if len(itemset) > 1:
                for i in range(1, len(itemset)):
                    antecedents = combinations(itemsets, i)
                    for antecedent in antecedents:
                        consequent = set(antecedent)
                        antecedent = set(itemsets) - consequent
                        antecedent_support = support_data.get(tuple(sorted(antecedent)), 0)
                        if antecedent_support > 0:  # Avoid division by zero
                            confidence = support / antecedent_support
                            rules.append({
                                "Consequent": sorted(antecedent),
                                "Antecedent": sorted(consequent),
                                "Confidence": round(confidence, 2)
                            })
                        else:
                            print(f"Skipping Rule {antecedent} -> {consequent} (Zero Antecedent Support)")
                else:
                    print(f"Skipping Itemset: {itemset} (Insufficient Items for Rules)")
    
    print("Generated Rules with Confidence:", rules)
    return rules

# Calculate confidence for all rules
rules_with_confidence = calculate_confidence(frequent_itemsets, support_data)

# Display Results
import pandas as pd
from IPython.display import display, HTML

rules_df = pd.DataFrame(rules_with_confidence)
if not rules_df.empty:
    print("Association Rules with Confidence:")
    display(HTML(rules_df.to_html(index=False)))
else:
    print("No rules were generated.")

{1: {'Non-Fatal Accidents <= 3000': 47, 'Killed <= 3000': 47, 'Injured <= 8000': 51, 'Total number of accidents <= 5000': 47, 'Total number of vehicles involved <= 5000': 43, 'Fatal Accidents <= 2000': 44}, 2: {('Injured <= 8000', 'Non-Fatal Accidents <= 3000'): 47, ('Injured <= 8000', 'Killed <= 3000'): 47, ('Killed <= 3000', 'Total number of accidents <= 5000'): 46, ('Injured <= 8000', 'Total number of accidents <= 5000'): 47}}
Skipping Rule {'Fatal Accidents <= 2000', 'Total number of vehicles involved <= 5000', 'Killed <= 3000', 'Injured <= 8000', 'Total number of accidents <= 5000'} -> {'Non-Fatal Accidents <= 3000'} (Zero Antecedent Support)
Skipping Rule {'Fatal Accidents <= 2000', 'Total number of vehicles involved <= 5000', 'Non-Fatal Accidents <= 3000', 'Injured <= 8000', 'Total number of accidents <= 5000'} -> {'Killed <= 3000'} (Zero Antecedent Support)
Skipping Rule {'Total number of vehicles involved <= 5000', 'Fatal Accidents <= 2000', 'Non-Fatal Accidents <= 3000', 'Kil

Consequent,Antecedent,Confidence
"[Injured <= 8000, Total number of accidents <= 5000]","[Fatal Accidents <= 2000, Killed <= 3000, Non-Fatal Accidents <= 3000, Total number of vehicles involved <= 5000]",1.00
"[Killed <= 3000, Total number of accidents <= 5000]","[Fatal Accidents <= 2000, Injured <= 8000, Non-Fatal Accidents <= 3000, Total number of vehicles involved <= 5000]",1.02
"[Injured <= 8000, Killed <= 3000]","[Fatal Accidents <= 2000, Non-Fatal Accidents <= 3000, Total number of accidents <= 5000, Total number of vehicles involved <= 5000]",1.00
"[Injured <= 8000, Non-Fatal Accidents <= 3000]","[Fatal Accidents <= 2000, Killed <= 3000, Total number of accidents <= 5000, Total number of vehicles involved <= 5000]",1.00
[Fatal Accidents <= 2000],"[Injured <= 8000, Killed <= 3000, Non-Fatal Accidents <= 3000, Total number of accidents <= 5000, Total number of vehicles involved <= 5000]",1.07
[Total number of vehicles involved <= 5000],"[Fatal Accidents <= 2000, Injured <= 8000, Killed <= 3000, Non-Fatal Accidents <= 3000, Total number of accidents <= 5000]",1.09
[Total number of accidents <= 5000],"[Fatal Accidents <= 2000, Injured <= 8000, Killed <= 3000, Non-Fatal Accidents <= 3000, Total number of vehicles involved <= 5000]",1.00
[Injured <= 8000],"[Fatal Accidents <= 2000, Killed <= 3000, Non-Fatal Accidents <= 3000, Total number of accidents <= 5000, Total number of vehicles involved <= 5000]",0.92
[Killed <= 3000],"[Fatal Accidents <= 2000, Injured <= 8000, Non-Fatal Accidents <= 3000, Total number of accidents <= 5000, Total number of vehicles involved <= 5000]",1.00
[Non-Fatal Accidents <= 3000],"[Fatal Accidents <= 2000, Injured <= 8000, Killed <= 3000, Total number of accidents <= 5000, Total number of vehicles involved <= 5000]",1.00
